In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("Online Retail Cleaned.csv")

In [6]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['Year']=df['InvoiceDate'].dt.year


In [7]:
df['Month']=df['InvoiceDate'].dt.month

In [9]:
df['Day']=df['InvoiceDate'].dt.day

In [10]:
df['Hour']=df['InvoiceDate'].dt.hour

In [15]:
df['Weekday']=df['InvoiceDate'].dt.day_name()

In [12]:
df['Day of week']=df['InvoiceDate'].dt.dayofweek

In [13]:
df['Quarter']=df['InvoiceDate'].dt.quarter

In [18]:
df['week of year']=df['InvoiceDate'].dt.isocalendar().week.astype(int)

In [19]:
df['Is Weekend']=df['Day of week'].isin([5,6]).astype(int)

# Customer level feature

In [22]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

customer_df = df.groupby('Customer ID').agg(
    Recency    = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency  = ('Invoice',     'nunique'),
    Monetary   = ('Total Price',  'sum'),
    AvgOrderValue   = ('Total Price', 'mean'),
    TotalItems      = ('Quantity',   'sum'),
    UniqueProducts  = ('StockCode',  'nunique'),
    FavoriteCountry = ('Country',    lambda x: x.mode()[0])
).reset_index()

print(customer_df.head())

   Customer ID  Recency  Frequency  Monetary  AvgOrderValue  TotalItems  \
0        12346      165         11    372.86      11.298788          70   
1        12347        3          2   1323.32      18.638310         828   
2        12348       74          1    222.16      11.108000         373   
3        12349       43          3   2630.34      26.042970         945   
4        12351       11          1    300.93      14.330000         261   

   UniqueProducts FavoriteCountry  
0              26  United Kingdom  
1              70         Iceland  
2              20         Finland  
3              89           Italy  
4              21     Unspecified  


# Time series feature

In [24]:
# Daily sales aggregate 
daily_sales = df.groupby(df['InvoiceDate'].dt.date).agg(
    Revenue      = ('Total Price', 'sum'),
    Orders       = ('Invoice',    'nunique'),
    ItemsSold    = ('Quantity',   'sum')
).reset_index()

daily_sales['InvoiceDate'] = pd.to_datetime(daily_sales['InvoiceDate'])
daily_sales = daily_sales.sort_values('InvoiceDate')

# Lag features — last day/week sales
daily_sales['Lag_1']  = daily_sales['Revenue'].shift(1)
daily_sales['Lag_7']  = daily_sales['Revenue'].shift(7)
daily_sales['Lag_30'] = daily_sales['Revenue'].shift(30)

# Rolling averages
daily_sales['RollingMean_7']  = daily_sales['Revenue'].rolling(7).mean()
daily_sales['RollingMean_30'] = daily_sales['Revenue'].rolling(30).mean()

# Nulls drop 
daily_sales = daily_sales.dropna()

# Churn lebel for churn prediction

In [25]:
# Last purchase date per customer
last_purchase = df.groupby('Customer ID')['InvoiceDate'].max().reset_index()
last_purchase.columns = ['Customer ID', 'LastPurchase']

snapshot_date = df['InvoiceDate'].max()
last_purchase['DaysSinceLastPurchase'] = (snapshot_date - last_purchase['LastPurchase']).dt.days

# 90 din se nahi aaya = churned (1), warna active (0)
last_purchase['Churned'] = (last_purchase['DaysSinceLastPurchase'] > 90).astype(int)

print(last_purchase['Churned'].value_counts())

# Customer features ke saath merge karo
customer_df = customer_df.merge(last_purchase[['Customer ID', 'Churned']], on='Customer ID')

Churned
0    2819
1    1407
Name: count, dtype: int64


# Product level feature

In [26]:
product_df = df.groupby('StockCode').agg(
    ProductName      = ('Description',  'first'),
    TotalSold        = ('Quantity',      'sum'),
    TotalRevenue     = ('Total Price',    'sum'),
    AvgPrice         = ('Price',         'mean'),
    OrderFrequency   = ('Invoice',       'nunique'),
    UniqueCustomers  = ('Customer ID',   'nunique')
).reset_index()

# Average daily demand
date_range = (df['InvoiceDate'].max() - df['InvoiceDate'].min()).days
product_df['AvgDailyDemand'] = product_df['TotalSold'] / date_range

# EOQ formula — Inventory optimization ke liye
# EOQ = sqrt(2 * D * S / H)
# D = annual demand, S = order cost (assume 10), H = holding cost (assume 2)
product_df['AnnualDemand'] = product_df['AvgDailyDemand'] * 365
product_df['EOQ'] = np.sqrt(
    (2 * product_df['AnnualDemand'] * 10) / 2
).round(0)

print(product_df.head())

  StockCode                  ProductName  TotalSold  TotalRevenue  AvgPrice  \
0     10002  INFLATABLE POLITICAL GLOBE        2616       2207.22  0.847222   
1     10080     GROOVY CACTUS INFLATABLE         12         10.20  0.850000   
2     10109         BENDY COLOUR PENCILS          4          1.68  0.420000   
3     10120                 DOGGY RUBBER        411         86.31  0.210000   
4    10123C        HEARTS WRAPPING TAPE         240        156.00  0.650000   

   OrderFrequency  UniqueCustomers  AvgDailyDemand  AnnualDemand    EOQ  
0             229              145        7.013405   2559.892761  160.0  
1               5                4        0.032172     11.742627   11.0  
2               1                1        0.010724      3.914209    6.0  
3              35               32        1.101877    402.184987   63.0  
4              41               34        0.643432    234.852547   48.0  


In [28]:
customer_df.to_csv("customer_features.csv",  index=False)
daily_sales.to_csv("daily_sales.csv",         index=False)
product_df.to_csv("product_features.csv",     index=False)

print("Feature engineering complete!")
print(f"Customers : {len(customer_df)}")
print(f"Daily rows: {len(daily_sales)}")
print(f"Products  : {len(product_df)}")

Feature engineering complete!
Customers : 4226
Daily rows: 277
Products  : 3989
